# Notebook 2 – Contact Stiffness Sweep

This notebook demonstrates how to use the `eb_cr_afm` solver to build the **calibration
curve** that is the backbone of quantitative CR-AFM: the mapping from measured contact
resonance frequency → tip–sample contact stiffness k₁ → sample Young's modulus E*.

## What you will learn

1. How to run a vectorised `contact_stiffness_sweep()` over a log-spaced range of k₁ values
2. How to visualise the **f_res(k₁)** and **Q(k₁)** dispersion curves interactively
3. How to build a **2D amplitude map** (k₁ × frequency) that shows all modes simultaneously
4. How to perform a simple **Hertz modulus inversion** from a measured f_res

## Background

In Hertz contact mechanics, the vertical contact stiffness relates to the reduced modulus E* as:

$$k_1 = 2 E^* a$$

where `a` is the contact radius. For a spherical tip of radius R in contact with
a flat sample indented by depth δ:  `a = √(R δ)`.

By inverting the f_res(k₁) calibration curve from this notebook, one can convert a
measured contact resonance frequency directly to a sample modulus value — without
any ad-hoc fitting.

**Probe:** Budget Sensors Multi 75-G  (k = 2.8 N/m, f_free ≈ 75 kHz)

All figures are saved to `../results/`.
This notebook is compatible with the Bradler-style axial tip offset parameter `cant.tip_offset`; when that offset is nonzero, the contact location is `cant.contact_x = L - tip_offset`.


In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '../src'))

import numpy as np
from eb_cr_afm import (
    default_params_multi75,
    ContactParams, SolverOptions,
    contact_stiffness_sweep,
    plot_stiffness_sweep_plotly,
    plot_stiffness_map_plotly,
)
from eb_cr_afm.spectra import compute_stiffness_frequency_map

# Results directory for saving figures
RESULTS_DIR = os.path.join(os.getcwd(), '../results')
os.makedirs(RESULTS_DIR, exist_ok=True)

cant, contact, exc, opts = default_params_multi75()
print(f'k = {cant.k:.1f} N/m   f_free ≈ 75 kHz')
print(f'Results will be saved to: {os.path.abspath(RESULTS_DIR)}')

## 1 – Solver options for sweeping

When sweeping many k₁ values we switch to the **analytical solver** (`method='analytical'`).
For the Multi 75-G probe the distributed electrostatic load `q(x) = 0` (the drive
is purely through sample surface displacement d₁), so the analytical particular solution
is identically zero and the 2×2 linear system reduces to a simple matrix inversion.
This makes the analytical solver **10–50× faster** than the iterative BVP solver with
no loss of accuracy.

**Frequency resolution:** `f_step=50.0 Hz` gives a very fine frequency grid (6800 points
over 60–400 kHz). Increase `f_step` (e.g. 500 Hz) for faster sweeps during development;
decrease for higher accuracy in Q-factor determination.

**k₁ range:** We sweep 25 log-spaced points from 1 N/m (near-free cantilever) to 800 N/m
(stiff silicon-like contact, k₁ ≈ 285 × k_cant). The log spacing gives good coverage of
the soft-contact, intermediate, and stiff-contact regimes.

In [ ]:
opts_fast = SolverOptions(
    n_points=100,
    f_start=60e3,
    f_stop=2000e3,
    f_step=50.0,
    g=20.0,
    method='analytical',   # q(x)=0 for Multi75 → exact particular solution,
                           # no BVP iteration needed; much faster for k1 sweeps
)

# k1 sweep: 1 N/m (barely in contact) → 1000 N/m (stiff Si-like)
k1_values = np.logspace(0, np.log10(100000), 25)  # 25 log-spaced
print(f'Sweeping {len(k1_values)} k₁ values from {k1_values[0]:.1f} to {k1_values[-1]:.0f} N/m')

## 2 – Run the contact stiffness sweep

`contact_stiffness_sweep()` loops over each k₁ value, runs `frequency_sweep()`,
and calls `sho_analysis()` to extract the dominant resonance peak parameters.

The result dict contains:
- `k1`: array of contact stiffness values (N/m)
- `f_res`: fitted resonance frequency for the dominant peak at each k₁ (Hz)
- `Q`: quality factor at each k₁
- `all_modes`: list of all detected modes per k₁ (useful for tracking higher modes)

**Expected behaviour:** As k₁ increases from 1 → 800 N/m, f_res increases monotonically
from ~75 kHz (free limit) toward the clamped-pinned limit (~329 kHz). Q initially
rises then plateaus as the peak narrows relative to the fixed air-damping background.

In [ ]:
stiff_sweep = contact_stiffness_sweep(
    k1_values,
    cant, contact, exc, opts_fast,
    k2_over_k1=0.0,
    verbose=True,
)
print('Done.')

## 3 – f_res and Q vs k₁ (interactive)

The two panels show:
- **Left:** Contact resonance frequency f_res (kHz) vs k₁ on a log axis.
  The curve is the calibration lookup table: measure f_res experimentally,
  read off k₁, then convert to E* via the Hertz model in Section 5.
- **Right:** Quality factor Q vs k₁. Higher Q means sharper peaks and more
  accurate frequency determination, but also longer ring-down times — a
  practical consideration for high-speed imaging.

Each eigenmode detected in the sweep window is plotted as a separate coloured trace.

In [ ]:
fig = plot_stiffness_sweep_plotly(stiff_sweep)
fig.update_layout(title='Multi 75-G: Contact Resonance vs k₁')
fig.show()

# Save figure
fig.write_html(os.path.join(RESULTS_DIR,  'nb02_fres_Q_vs_k1.html'))
print('Saved: nb02_fres_Q_vs_k1.html')

## 4 – 2D amplitude map: amplitude(k₁, f)

`compute_stiffness_frequency_map()` computes the full frequency spectrum at every
k₁ value and stores the results as a 2D array indexed by (k₁ index, frequency index).

The resulting heatmap (colour = log amplitude) shows:
- The **contact resonance dispersion ridge** — the high-amplitude band that tracks
  the shifting resonance peak with k₁
- The **off-resonance background** — much lower amplitude, revealing how efficiently
  the drive couples to the beam at other frequencies
- Dashed guide lines marking the free-cantilever stiffness (`k₁ = k`) and
  the stiff-contact limit (`k₁ = 100k`)

The heatmap format is ideal for identifying multiple eigenmodes simultaneously,
since all resonance peaks in the sweep window appear as separate parallel ridges.

In [ ]:
sf_map = compute_stiffness_frequency_map(
    cant, contact, exc, opts_fast,
    k1_values=k1_values,
    k2_over_k1=0.0,
    verbose=True,
)
print('2D map done.')

In [ ]:
fig = plot_stiffness_map_plotly(sf_map, cant=cant)
fig.update_layout(title='Multi 75-G: 2D Spectra (k₁ vs Frequency)')
fig.show()

# Save figure
fig.write_html(os.path.join(RESULTS_DIR,  'nb02_2d_amplitude_map.html'))
print('Saved: nb02_2d_amplitude_map.html')

## 5 – k₁ → sample modulus inversion (Hertz contact)

Once the f_res(k₁) calibration curve is in hand, converting a measured contact
resonance frequency to sample modulus E* is a two-step lookup:

**Step 1:** Interpolate the calibration curve to find k₁ from f_res.

**Step 2:** Apply the Hertz contact model:
$$k_1 = 2 E^* a, \qquad a = \sqrt{R \delta}$$

where R is the tip radius and δ is the indentation depth (typically 1–2 nm for
light-tapping CR-AFM). The reduced modulus E* combines tip (t) and sample (s):
$$\frac{1}{E^*} = \frac{1-\nu_s^2}{E_s} + \frac{1-\nu_t^2}{E_t}$$

For a silicon tip (E_t = 170 GPa, ν_t = 0.27), the tip compliance is ~2 % of E*
for polymer samples and negligible for hard materials.

**Note:** Replace `R_tip_nm` and `delta_nm` with your actual experimental parameters.
The contact model (Hertz, DMT, JKR) should match your tip geometry and loading conditions.

In [ ]:
# Example inversion for a measured f_res
f_measured_kHz = 220.0    # kHz — your experimental value
R_tip_nm       = 10.0     # nm tip radius
delta_nm       = 1.0      # nm indentation depth

# Interpolate k1 from the sweep at the measured frequency
from scipy.interpolate import interp1d
interp_k1 = interp1d(stiff_sweep['f_res'] / 1e3, stiff_sweep['k1'],
                      kind='linear', fill_value='extrapolate')
k1_estimated = float(interp_k1(f_measured_kHz))

# Hertz modulus
a_contact = np.sqrt(R_tip_nm * 1e-9 * delta_nm * 1e-9)
E_star = k1_estimated / (2 * a_contact)

print(f'f_measured = {f_measured_kHz:.1f} kHz')
print(f'k1 (from sweep) = {k1_estimated:.1f} N/m')
print(f'E* (Hertz, a = sqrt(R·δ)) = {E_star/1e9:.2f} GPa')